# 基于并行广度优先遍历的网络故障诊断实验

广度优先遍历以层次扩展方式访问图中节点，计算过程需要频繁读取邻接表并更新访问状态，因此能够体现图数据布局、邻接关系访问、片上缓存利用和多核任务划分对图计算性能的影响。本实验围绕网络故障后的替换结点搜索展开，在Ascend C环境中使用静态Tensor编程方式实现并行层次扩展算子。

本实验学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建`Source/02.01`实验工程并加载CANN环境；
3. 问题分析：说明输入输出、数据布局、分块策略、多核方式和实验参数；
4. 核函数开发：实现基于静态Tensor分块的并行层次扩展核函数；
5. 结果验证与性能分析：准备Host输入和CPU参考结果，开发核函数调用代码，完成构建、运行、结果校验和性能分析；
6. 实验总结：归纳CSR存储、片上分块、多核扩展和层次维护的实现过程。


---
## 1. 实验概述

本实验以数据中心服务器集群故障替换为具体场景，围绕图结构、CSR压缩表示和并行广度优先遍历展开。实验将同一数据中心内的服务器集群抽象为图结构：结点表示服务器，边表示服务器之间的通信链路。服务器状态分为故障和完好两类，除故障服务器外，完好服务器均可作为临时替换对象。当某台服务器发生故障时，需要从该服务器出发，按照集群拓扑跳数逐层搜索，找到距离最近的完好服务器。需要在Ascend C环境中采用静态Tensor编程方式组织并行遍历过程，完成服务器集群拓扑构造、CSR存储转换、Kernel启动和结果写回，并以Host侧串行结果作为正确性参考。通过本任务，能够理解图结构在Device侧的组织方法，并能够从正确性、处理边数、有效边处理吞吐率和加速比等方面分析实验结果。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解服务器集群图的数据组织关系。掌握图结构、邻接表和CSR压缩表示之间的关系，理解服务器连接关系如何由分散的邻接记录转换为连续数组结构，能够说明故障服务器、候选服务器、通信链路、访问状态和BFS层次之间的对应关系，并理解CSR行指针数组和邻接结点数组在遍历过程中的作用。
2. 掌握Ascend C并行广度优先遍历实现的基本方法。能够理解Host侧和Device侧在实验中的分工：Host侧负责拓扑生成、CSR构建、运行时管理、Device内存申请、Kernel启动、每个核的位图合并、访问状态维护、层次编号维护和下一层frontier生成；Device侧负责并行扩展当前frontier的邻接关系，并将发现的邻接结点写入每核私有位图。通过实验，应能够理解静态Tensor分块、LocalTensor数据暂存、多AI Core任务划分和Host侧层次推进之间的配合关系。
3. 具备正确性验证和性能分析能力。能够将Device侧结果与Host侧参考结果进行对比，判断最近完好服务器、最短跳数和访问状态是否一致；能够根据Kernel耗时、总耗时、有效边处理吞吐率和加速比分析不同集群拓扑与分块设置对性能的影响。


### 1.2 前置知识

本实验案例在实验前应先理解服务器集群拓扑表示和广度优先遍历过程，再学习CSR压缩存储、静态Tensor编程方式、数据搬运和多核任务划分。建议在实验前重点掌握如下内容：

1. 图结构与网络拓扑基础：理解图结点和连通路径等基本概念，明确数据中心服务器故障替换问题可以抽象为从故障服务器出发寻找可达完好服务器的问题。实验中的最近不是物理距离意义上的最近，而是服务器拓扑中的最短跳数。
2. 邻接表与CSR压缩表示基础：理解邻接表适合服务器集群图的直观构造，CSR适合连续数组访问。行指针数组用于保存各结点邻接范围的边界，邻接结点数组用于保存相邻结点编号，二者共同表达每个结点的邻接关系。
3. 广度优先遍历与层次推进基础：理解广度优先遍历按照层次逐轮扩展。当前层待扩展结点集合表示本轮需要处理的结点，下一层待访问结点集合表示下一轮继续处理的结点。对于等权图，首次发现的完好候选服务器具有最小跳数。
4. Ascend C开发基础：Host侧负责服务器集群拓扑生成、CSR构建、运行时管理、Device侧内存申请、Kernel启动和结果校验；Device侧执行Kernel核函数，完成邻接关系访问、访问状态维护以及候选结果写回。实验前应熟悉工程编译与脚本运行方法，同时掌握性能指标查看方法。
5. 静态Tensor编程基础：理解静态Tensor编程方式要求开发者显式规划数据块边界、LocalTensor地址和存储位置，并根据数据搬入、计算和写回之间的依赖关系管理同步。理解图结构数据通常先从Global Memory搬入Local Memory，再在片上完成局部处理，最后写回Global Memory。


### 1.3 实验要点

实验中应重点关注以下内容：

1. 服务器集群场景建模：根据实验规模生成服务器及其通信链路，设置服务器健康状态和故障起点。在完好服务器均可替换的前提下，将故障替换问题转化为最近完好服务器搜索问题。
2. 图结构转换：先使用邻接表组织每个结点的邻接关系，再根据结点度数形成CSR压缩表示，确保Host侧和Device侧对结点数量、边数量和邻接范围的理解一致。
3. 层次推进：以故障服务器为起点生成第一层待扩展结点集合，每一轮处理当前层结点的邻接结点，维护访问状态和层次编号，并生成下一层待访问结点集合。
4. 数据搬运与静态Tensor分块组织：将当前层结点块和邻接关系片段从Global Memory搬入Local Memory，在LocalTensor中完成访问状态判断和候选结果记录，说明数据块边界、LocalTensor存储位置与结果写回范围之间的关系。
5. 候选结果判定：遍历过程中检查邻接结点健康状态。当某一层出现完好候选服务器时，该层结点在跳数意义上距离故障服务器最近；如果存在多个候选服务器，应按照编号或预设优先级确定最终结果。
6. 结果正确性验证：使用Host侧参考结果对Device侧输出进行比较，结合最近完好服务器、最短跳数和访问状态判断结果是否可信。
7. 性能结果分析：记录预处理时间、Kernel耗时、总耗时、有效边处理吞吐率和加速比，并结合服务器集群拓扑解释实验现象。


---
## 2. 环境准备

首先创建实验所需目录，并尝试加载Ascend CANN环境变量。

目录划分如下：

- `Source/02.01/include`：保存Host侧公共参数、数据结构和辅助函数。
- `Source/02.01/src`：保存网络拓扑生成和CPU参考计算代码。
- `Source/02.01/ascend_ops/op_kernel`：保存Device侧并行层次扩展核函数。
- `Source/02.01/ascend_ops/host_launch`：保存Host侧核函数调用与结果校验代码。
- `Source/02.01/scripts`：保存完整构建、运行和性能分析脚本。
- `Source/02.01/results`：保存每次运行得到的CSV实验结果。
- `Source/02.01/CMakeLists.txt`：配置Host程序与Device核函数的构建流程。

下面的代码会创建这些目录、搜索常见的Ascend安装位置，并确认编译工具链是否可用。如果当前环境没有安装CANN，仍然可以继续阅读和生成全部代码，但完整编译和运行需要在已配置Ascend设备与CANN工具链的环境中完成。


In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess

PROJECT_ROOT = Path("Source/02.01")
for directory in [
    "include",
    "src",
    "scripts",
    "results",
    "ascend_ops/op_kernel",
    "ascend_ops/host_launch",
]:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

def find_cann_root(env_script):
    for parent in [env_script.parent, *env_script.parents]:
        cmake_candidates = [
            parent / "tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
        ]
        if any(path.is_file() for path in cmake_candidates):
            return parent
    return None

candidate_scripts = []
for item in [
    os.environ.get("ASCEND_TOOLKIT_HOME"),
    os.environ.get("ASCEND_INSTALL_PATH"),
    "/usr/local/Ascend/ascend-toolkit/latest",
]:
    if item:
        candidate_scripts.append(Path(item) / "set_env.sh")

for root in [Path("/usr/local/Ascend"), Path("/home/ma-user/Ascend"), Path("/opt/Ascend"), Path("/opt/conda/Ascend")]:
    if root.exists():
        candidate_scripts.extend(sorted(root.glob("**/set_env.sh"), reverse=True))

set_env = None
install_root = None
for script in candidate_scripts:
    if not script.is_file():
        continue
    root = find_cann_root(script)
    if root is not None:
        set_env = script
        install_root = root
        break

if set_env is not None and shutil.which("bash"):
    command = f"source {shlex.quote(str(set_env))} && env"
    loaded_env = subprocess.check_output(
        ["bash", "-lc", command],
        text=True,
    )
    for line in loaded_env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_INSTALL_PATH"] = str(install_root)
    os.environ["ASCEND_CANN_PACKAGE_PATH"] = str(install_root)
    print("Ascend environment loaded from:", set_env)
    print("ASCEND_INSTALL_PATH:", install_root)
else:
    print("Ascend set_env.sh was not found. Code generation can continue; build/run needs CANN.")
    print("Diagnostic command: !find /usr/local /home/ma-user /opt -name set_env.sh 2>/dev/null")

print("Experiment directory:", PROJECT_ROOT.resolve())
print("cmake:", shutil.which("cmake") or "not found")
print("C++ compiler:", shutil.which("g++") or shutil.which("c++") or "not found")


---
## 3. 问题分析

### 3.1 计算目标

网络故障诊断需要从故障结点出发，按照通信链路逐层搜索最近的健康结点。若同一最短距离层中存在多个健康结点，则选择编号最小的结点，使结果稳定且便于重复验证。

本实验把完整广度优先遍历拆成多次单层扩展。一次算子调用只负责读取当前层结点并发现其所有邻接结点；Host负责在层与层之间合并发现结果、排除已访问结点并决定是否继续搜索。

设健康结点集合为$H$，故障结点为$s$。最短距离定义为：

$$
d_{\min}=\min_{v\in H}\operatorname{dist}(s,v)
$$

最终候选是最短距离层中编号最小的健康结点：

$$
replacement=\min\left\{v\in H\mid \operatorname{dist}(s,v)=d_{\min}\right\}
$$


### 3.2 输入、输出与数据布局

算子的主要输入由三部分组成：

- 网络拓扑：使用CSR压缩存储，由行偏移信息和连续邻接结点信息共同描述；
- 当前层结点集合：保存本轮需要扩展的结点；
- 运行参数：描述图规模、当前层有效长度、片上分块长度和并行核心数。

每个AI Core输出一份私有发现位图和处理边数。位图使用1bit表示一个结点是否被发现。设结点数为$N$，单核位图存储量为：

$$
S_{\mathrm{bitmap,core}}=\left\lceil\frac{N}{32}\right\rceil\times4\ \mathrm{bytes}
$$


### 3.3 静态Tensor分块与多核策略

当前层结点先按固定长度切分为多个块，再把连续块分配给不同AI Core。每个核心处理自己负责的结点块，并将较长的邻接区间继续切分为固定长度片段。

核函数为当前层结点和邻接片段分别预留固定容量的片上存储。实际运行时只处理本次分块中的有效元素，尾部不足一个完整块时使用安全填充完成整块搬运。

设并行核心数为$P$。每个核心使用独立位图记录发现结果，算子结束后由Host执行按位或归并：

$$
B_{\mathrm{merged}}=\bigvee_{c=0}^{P-1}B_c
$$

归并后的位图经过已访问过滤后形成下一层结点集合。这样既能让多个AI Core并行访问邻接边，又能保持广度优先遍历逐层推进的顺序。


### 3.4 公共参数定义

前面分析的图规模、故障条件、分块大小、并行核心数和测量次数需要在CPU参考与NPU运行之间保持一致。下面的公共代码集中定义这些实验参数，以及网络图、遍历结果、计时信息和正确性检查所需的数据结构。

将公共参数放在同一处，可以保证两条计算路径使用相同的输入条件，也便于后续扩展实验统一修改参数。


In [ ]:
%%writefile Source/02.01/include/network_bfs_common.h

#pragma once

#include <algorithm>
#include <chrono>
#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <stdexcept>
#include <string>
#include <vector>

namespace netbfs {

constexpr int32_t kUnvisited = -1;
constexpr int32_t kNoCandidate = -1;
constexpr int32_t kDeviceInvalidCandidate = 0x7fffffff;
constexpr uint32_t kStaticMaxFrontierBlockLen = 2048;
constexpr uint32_t kStaticMaxNeighborTileLen = 2048;

struct BfsConfig {
    uint32_t num_nodes = 262144;
    uint32_t avg_degree = 32;
    uint32_t seed = 1234;
    uint32_t fault_node = 0;
    uint32_t health_percent = 1;
    // Nodes with BFS distance <= fault_radius from fault_node are forced unavailable.
    // This creates deterministic multi-level diagnosis cases for performance tests.
    uint32_t fault_radius = 4;
    uint32_t frontier_block_len = 32;
    uint32_t neighbor_tile_len = 32;
    uint32_t block_dim = 4;
    uint32_t warmup = 2;
    uint32_t repeat = 10;
    uint32_t max_depth = 0;  // 0 means num_nodes.
    bool sweep = false;
    bool print_graph = false;
    bool print_levels = false;
};

struct CsrGraph {
    uint32_t num_nodes = 0;
    uint32_t undirected_edges = 0;
    std::vector<int32_t> row_ptr;
    std::vector<int32_t> col_idx;
    std::vector<int32_t> status;  // 1 means healthy and replaceable; 0 means faulty/unavailable.
};

struct TimingUs {
    double preprocess_us = 0.0;
    double bfs_total_us = 0.0;
    double kernel_us = 0.0;
    double host_frontier_us = 0.0;
};

struct BfsResult {
    int32_t replacement_node = kNoCandidate;
    int32_t distance = -1;
    uint32_t visited_count = 0;
    uint64_t processed_edges = 0;
    std::vector<int32_t> level;
    std::vector<uint32_t> frontier_sizes;
    std::vector<uint64_t> frontier_edges;
    TimingUs timing;
};

struct CheckResult {
    bool replacement_ok = false;
    bool distance_ok = false;
    bool level_ok = false;
    uint32_t level_mismatch = 0;
};

class Timer {
public:
    Timer() : start_(std::chrono::high_resolution_clock::now()) {}
    double elapsed_us() const {
        const auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(end - start_).count();
    }
private:
    std::chrono::high_resolution_clock::time_point start_;
};

inline uint32_t effective_max_depth(const BfsConfig& cfg) {
    return cfg.max_depth == 0 ? cfg.num_nodes : cfg.max_depth;
}

inline uint32_t round_up(uint32_t x, uint32_t align) {
    if (align == 0) return x;
    return ((x + align - 1) / align) * align;
}

inline void check_config(const BfsConfig& cfg) {
    if (cfg.num_nodes == 0) {
        throw std::invalid_argument("--nodes must be positive");
    }
    if (cfg.fault_node >= cfg.num_nodes) {
        throw std::invalid_argument("--fault-node must be in [0, nodes)");
    }
    if (cfg.avg_degree == 0) {
        throw std::invalid_argument("--avg-degree must be positive");
    }
    if (cfg.health_percent > 100) {
        throw std::invalid_argument("--health-percent must be in [0, 100]");
    }
    if (cfg.frontier_block_len == 0 || cfg.frontier_block_len > kStaticMaxFrontierBlockLen) {
        throw std::invalid_argument("--frontier-block-len must be in [1, 2048]");
    }
    if (cfg.neighbor_tile_len == 0 || cfg.neighbor_tile_len > kStaticMaxNeighborTileLen) {
        throw std::invalid_argument("--neighbor-tile-len must be in [1, 2048]");
    }
    if ((cfg.frontier_block_len * sizeof(int32_t)) % 32 != 0) {
        throw std::invalid_argument("frontier_block_len*sizeof(int32_t) must be 32B aligned");
    }
    if ((cfg.neighbor_tile_len * sizeof(int32_t)) % 32 != 0) {
        throw std::invalid_argument("neighbor_tile_len*sizeof(int32_t) must be 32B aligned");
    }
    if (cfg.block_dim == 0) {
        throw std::invalid_argument("--block-dim must be positive");
    }
}


公共辅助代码负责检查参数范围、比较CPU和Device结果，并统一输出网络规模和性能数据。分块长度需要满足片上容量与搬运对齐要求，正确性检查则比较候选结点、最短距离和完整层次数组。


In [ ]:
%%writefile -a Source/02.01/include/network_bfs_common.h

inline uint64_t directed_edge_count(const CsrGraph& g) {
    return static_cast<uint64_t>(g.col_idx.size());
}

inline double edges_per_second(uint64_t edges, double us) {
    if (us <= 0.0) return 0.0;
    return static_cast<double>(edges) / (us * 1e-6);
}

inline CheckResult compare_results(const BfsResult& got, const BfsResult& ref) {
    CheckResult c;
    c.replacement_ok = (got.replacement_node == ref.replacement_node);
    c.distance_ok = (got.distance == ref.distance);
    c.level_ok = (got.level.size() == ref.level.size());
    if (c.level_ok) {
        for (size_t i = 0; i < got.level.size(); ++i) {
            if (got.level[i] != ref.level[i]) {
                ++c.level_mismatch;
                c.level_ok = false;
            }
        }
    }
    return c;
}

inline void print_graph_summary(const CsrGraph& g, uint32_t fault_node) {
    uint32_t healthy = 0;
    for (int32_t s : g.status) healthy += (s == 1 ? 1u : 0u);
    std::cout << "graph: nodes=" << g.num_nodes
              << ", directed_edges=" << g.col_idx.size()
              << ", undirected_edges=" << g.undirected_edges
              << ", healthy_nodes=" << healthy
              << ", fault_node=" << fault_node << "\n";
}

inline void print_levels_sample(const BfsResult& result, size_t count = 32) {
    const size_t n = std::min(result.level.size(), count);
    std::cout << "level sample(first " << n << "):\n";
    for (size_t i = 0; i < n; ++i) {
        std::cout << "  node=" << i << " level=" << result.level[i] << "\n";
    }
}

inline void print_result_header(bool ascend) {
    std::cout << std::setw(10) << "nodes"
              << std::setw(12) << "dirEdges"
              << std::setw(12) << "fault"
              << std::setw(12) << "replace"
              << std::setw(10) << "dist"
              << std::setw(12) << "visited"
              << std::setw(14) << "procEdges"
              << std::setw(14) << (ascend ? "kernel_us" : "bfs_us")
              << std::setw(14) << (ascend ? "frontier_us" : "prep_us")
              << std::setw(14) << "total_us"
              << std::setw(14) << "MEdges/s"
              << std::setw(12) << "status"
              << "\n";
}

inline void print_result_row(const CsrGraph& g,
                             uint32_t fault_node,
                             const BfsResult& result,
                             const std::string& status,
                             bool ascend) {
    const double time_for_rate = ascend ? result.timing.bfs_total_us : result.timing.bfs_total_us;
    const double medges = edges_per_second(result.processed_edges, time_for_rate) / 1e6;
    std::cout << std::setw(10) << g.num_nodes
              << std::setw(12) << g.col_idx.size()
              << std::setw(12) << fault_node
              << std::setw(12) << result.replacement_node
              << std::setw(10) << result.distance
              << std::setw(12) << result.visited_count
              << std::setw(14) << result.processed_edges
              << std::setw(14) << std::fixed << std::setprecision(2)
              << (ascend ? result.timing.kernel_us : result.timing.bfs_total_us)
              << std::setw(14) << (ascend ? result.timing.host_frontier_us : result.timing.preprocess_us)
              << std::setw(14) << result.timing.bfs_total_us
              << std::setw(14) << std::setprecision(3) << medges
              << std::setw(12) << status
              << std::defaultfloat
              << "\n";
}

}  // namespace netbfs


---
## 4. 核函数开发

### 4.1 多核任务划分

算子按照当前层数据块数量和启动核心数划分任务。每个核心领取一段连续数据块；当数据块数量不能整除核心数时，尾部核心处理较少任务，没有任务的核心直接结束。

每个核心拥有独立的发现位图，因此多个核心可以同时记录结果，不需要对共享访问状态进行原子更新。


In [ ]:
%%writefile Source/02.01/ascend_ops/op_kernel/bfs_expand_level.cpp

// Experiment 2 kernel: one BFS level expansion over a CSR graph.
//
// Host-merged per-core bitset design:
//   - current frontier is stored as a dense int32 array in GM;
//   - CSR row_ptr and col_idx encode adjacency lists;
//   - each AI Core owns a contiguous range of frontier blocks;
//   - static UB tensors stage frontier nodes and neighbor segments;
//   - the kernel writes a per-core bitset slice for discovered neighbor
//     vertices, using one bit per graph vertex;
//   - Host OR-reduces the per-core bitsets and performs the canonical
//     visited/level update.
//
// Private per-core bitsets provide race-free discovery while reducing the
// returned next-frontier state from blockDim*numNodes int32 flags to
// blockDim*ceil(numNodes/32) compact bitset words.

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kStaticMaxFrontierBlockLen = 2048;
constexpr uint32_t kStaticMaxNeighborTileLen = 2048;
constexpr int32_t kDeviceInvalidCandidate = 0x7fffffff;

__aicore__ inline void GetBlockRange(uint32_t numBlocks,
                                      uint32_t launchBlockDim,
                                      uint32_t& beginBlock,
                                      uint32_t& endBlock) {
    const uint32_t coreNum = launchBlockDim == 0 ? 1 : launchBlockDim;
    const uint32_t coreId = GetBlockIdx();
    if (coreId >= coreNum) {
        beginBlock = 0;
        endBlock = 0;
        return;
    }
    const uint32_t blocksPerCore = (numBlocks + coreNum - 1) / coreNum;
    beginBlock = coreId * blocksPerCore;
    endBlock = beginBlock + blocksPerCore;
    if (endBlock > numBlocks) {
        endBlock = numBlocks;
    }
}

__aicore__ inline uint32_t MinU32(uint32_t a, uint32_t b) {
    return a < b ? a : b;
}
}  // namespace


### 4.2 全局数据绑定与边界检查

核函数首先建立对输入输出数据的全局视图，并根据图规模计算位图长度。随后检查分块长度、当前层规模和核心编号，确保本次任务处于有效范围。

这一步只负责建立数据范围和执行边界，不进行实际邻接遍历。


In [ ]:
%%writefile -a Source/02.01/ascend_ops/op_kernel/bfs_expand_level.cpp

extern "C" __global__ __aicore__ void bfs_expand_level(GM_ADDR rowPtr,
                                                        GM_ADDR colIdx,
                                                        GM_ADDR status,
                                                        GM_ADDR frontier,
                                                        GM_ADDR visited,
                                                        GM_ADDR level,
                                                        GM_ADDR nextBitmap,
                                                        GM_ADDR candidatePerCore,
                                                        GM_ADDR edgeCountPerCore,
                                                        uint32_t numNodes,
                                                        uint32_t numEdgesPadded,
                                                        uint32_t frontierCount,
                                                        uint32_t frontierPaddedCount,
                                                        uint32_t currentDepth,
                                                        uint32_t frontierBlockLen,
                                                        uint32_t neighborTileLen,
                                                        uint32_t launchBlockDim,
                                                        uint32_t faultNode) {
    InitSocState();

    GlobalTensor<int32_t> rowPtrGm;
    GlobalTensor<int32_t> colIdxGm;
    GlobalTensor<int32_t> frontierGm;
    GlobalTensor<int32_t> nextBitmapGm;
    GlobalTensor<int32_t> candidateGm;
    GlobalTensor<int32_t> edgeCountGm;

    rowPtrGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(rowPtr), numNodes + 1);
    colIdxGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(colIdx), numEdgesPadded);
    frontierGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(frontier), frontierPaddedCount);
    const uint32_t wordsPerBitmap = (numNodes + 31u) >> 5;
    nextBitmapGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(nextBitmap), wordsPerBitmap * launchBlockDim);
    candidateGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(candidatePerCore), launchBlockDim);
    edgeCountGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(edgeCountPerCore), launchBlockDim);

    const uint32_t coreId = GetBlockIdx();
    if (coreId >= launchBlockDim) {
        return;
    }
    candidateGm.SetValue(coreId, kDeviceInvalidCandidate);
    edgeCountGm.SetValue(coreId, 0);
    PipeBarrier<PIPE_ALL>();

    if (frontierBlockLen == 0 || frontierBlockLen > kStaticMaxFrontierBlockLen ||
        neighborTileLen == 0 || neighborTileLen > kStaticMaxNeighborTileLen ||
        frontierCount == 0 || numNodes == 0 || numEdgesPadded == 0) {
        return;
    }

    const uint32_t numFrontierBlocks = (frontierCount + frontierBlockLen - 1) / frontierBlockLen;
    uint32_t beginBlock = 0;
    uint32_t endBlock = 0;
    GetBlockRange(numFrontierBlocks, launchBlockDim, beginBlock, endBlock);
    if (beginBlock >= endBlock) {
        return;
    }


### 4.3 片上分块与邻接扩展

每个核心在片上准备两个固定容量的缓冲区，分别缓存当前层结点块和邻接片段。处理过程中先搬入一块当前层结点，再根据每个结点的邻接范围分段搬运邻接数据。

发现有效邻接结点后，核心在自己的位图区域设置对应标记。所有任务完成后，将发现位图和处理边数写回Global Memory。


In [ ]:
%%writefile -a Source/02.01/ascend_ops/op_kernel/bfs_expand_level.cpp

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<int32_t> frontierLocal = ubAllocator.Alloc<int32_t, kStaticMaxFrontierBlockLen>();
    LocalTensor<int32_t> neighborLocal = ubAllocator.Alloc<int32_t, kStaticMaxNeighborTileLen>();
    frontierLocal.SetSize(kStaticMaxFrontierBlockLen);
    neighborLocal.SetSize(kStaticMaxNeighborTileLen);

    int32_t processedEdges = 0;  // debug-only; Host computes the authoritative value.

    for (uint32_t blockId = beginBlock; blockId < endBlock; ++blockId) {
        const uint32_t frontierBase = blockId * frontierBlockLen;
        const uint32_t activeFrontierLen = MinU32(frontierBlockLen, frontierCount - frontierBase);
        DataCopy(frontierLocal, frontierGm[frontierBase], frontierBlockLen);
        PipeBarrier<PIPE_ALL>();

        for (uint32_t i = 0; i < activeFrontierLen; ++i) {
            const int32_t u = frontierLocal.GetValue(i);
            if (u < 0 || static_cast<uint32_t>(u) >= numNodes) {
                continue;
            }
            const int32_t begin = rowPtrGm.GetValue(static_cast<uint32_t>(u));
            const int32_t end = rowPtrGm.GetValue(static_cast<uint32_t>(u) + 1);
            if (end <= begin) {
                continue;
            }
            processedEdges += (end - begin);

            uint32_t edgePos = static_cast<uint32_t>(begin);
            const uint32_t edgeEnd = static_cast<uint32_t>(end);
            while (edgePos < edgeEnd) {
                const uint32_t activeNeighborLen = MinU32(neighborTileLen, edgeEnd - edgePos);
                DataCopy(neighborLocal, colIdxGm[edgePos], neighborTileLen);
                PipeBarrier<PIPE_ALL>();

                for (uint32_t j = 0; j < activeNeighborLen; ++j) {
                    const int32_t v = neighborLocal.GetValue(j);
                    if (v < 0 || static_cast<uint32_t>(v) >= numNodes) {
                        continue;
                    }
                    // Per-core bitset layout:
                    // nextBitmap[coreId * wordsPerBitmap + (vertex / 32)].
                    // Each AI Core owns a private slice, so there is no cross-core
                    // write race. Host later OR-reduces these slices and assigns the
                    // canonical BFS level for the next frontier.
                    const uint32_t vu = static_cast<uint32_t>(v);
                    const uint32_t wordOffset = vu >> 5;
                    const uint32_t bitMask = 1u << (vu & 31u);
                    const uint32_t bitmapIndex = coreId * wordsPerBitmap + wordOffset;
                    const uint32_t oldWord = static_cast<uint32_t>(nextBitmapGm.GetValue(bitmapIndex));
                    nextBitmapGm.SetValue(bitmapIndex, static_cast<int32_t>(oldWord | bitMask));
                }
                edgePos += activeNeighborLen;
            }
        }
    }

    edgeCountGm.SetValue(coreId, processedEdges);

    (void)status;
    (void)visited;
    (void)level;
    (void)currentDepth;
    (void)faultNode;
}


### 4.4 层次结果合并

算子结束后，Host读取各核心的发现位图并进行按位合并。新发现结点经过已访问过滤后被赋予下一层层次编号，同时形成下一层待处理结点集合。

如果本层包含健康结点，则选择其中编号最小的结点作为最终候选；否则继续启动下一层算子。


---
## 5. 结果验证与性能分析

### 5.1 Host数据准备与CPU参考

Host生成可重复的网络拓扑、设置结点状态并转换为CSR。CPU串行遍历在NPU程序内部执行，为候选结点、最短距离和完整层次数组提供正确性参考。


#### 5.1.1 CPU参考接口

CPU参考接口提供网络图生成、串行遍历和小规模图查看功能。它与NPU Host共享第二部分定义的配置和结果结构。


In [ ]:
%%writefile Source/02.01/include/network_bfs_cpu.h

#pragma once

#include "network_bfs_common.h"

namespace netbfs {

CsrGraph make_network_graph(const BfsConfig& cfg);
BfsResult serial_bfs_reference(const CsrGraph& g, uint32_t fault_node, uint32_t max_depth = 0);
void dump_csr_small(const CsrGraph& g, uint32_t max_nodes = 16);

}  // namespace netbfs


#### 5.1.2 网络图生成

图生成时，先将相邻编号结点依次连接，并让首尾结点相连，以保证整个网络连通；随后添加随机无向边达到目标平均度。邻接关系排序去重后转换为CSR，并根据健康比例和不可用范围生成结点状态。


In [ ]:
%%writefile Source/02.01/src/network_bfs_cpu.cpp

#include "network_bfs_cpu.h"

#include <deque>
#include <random>
#include <unordered_set>

namespace netbfs {
namespace {

uint64_t edge_key(uint32_t u, uint32_t v) {
    if (u > v) std::swap(u, v);
    return (static_cast<uint64_t>(u) << 32) | static_cast<uint64_t>(v);
}

void add_undirected_edge(uint32_t u,
                         uint32_t v,
                         std::vector<std::vector<int32_t>>& adj,
                         std::unordered_set<uint64_t>& used) {
    if (u == v) return;
    const uint64_t key = edge_key(u, v);
    if (!used.insert(key).second) return;
    adj[u].push_back(static_cast<int32_t>(v));
    adj[v].push_back(static_cast<int32_t>(u));
}

std::vector<int32_t> collect_nodes_within_radius(const std::vector<std::vector<int32_t>>& adj,
                                                 uint32_t source,
                                                 uint32_t radius) {
    const uint32_t n = static_cast<uint32_t>(adj.size());
    std::vector<int32_t> depth(n, -1);
    std::deque<uint32_t> q;
    depth[source] = 0;
    q.push_back(source);
    while (!q.empty()) {
        const uint32_t u = q.front();
        q.pop_front();
        if (static_cast<uint32_t>(depth[u]) >= radius) continue;
        for (int32_t vv : adj[u]) {
            if (vv < 0) continue;
            const uint32_t v = static_cast<uint32_t>(vv);
            if (v >= n || depth[v] >= 0) continue;
            depth[v] = depth[u] + 1;
            q.push_back(v);
        }
    }
    return depth;
}

}  // namespace

CsrGraph make_network_graph(const BfsConfig& cfg) {
    check_config(cfg);
    Timer timer;

    const uint32_t n = cfg.num_nodes;
    std::vector<std::vector<int32_t>> adj(n);
    std::unordered_set<uint64_t> used;
    used.reserve(static_cast<size_t>(n) * std::max(2u, cfg.avg_degree));

    // Connect consecutive nodes in a ring so the graph stays connected.
    // Random extra links then create nonuniform degrees and realistic frontiers.
    if (n >= 2) {
        for (uint32_t u = 0; u < n; ++u) {
            add_undirected_edge(u, (u + 1) % n, adj, used);
        }
    }

    const uint64_t target_undirected_edges = std::max<uint64_t>(
        used.size(),
        std::max<uint64_t>(1, (static_cast<uint64_t>(n) * cfg.avg_degree) / 2));

    std::mt19937 rng(cfg.seed);
    std::uniform_int_distribution<uint32_t> node_dist(0, n - 1);
    uint64_t guard = 0;
    const uint64_t guard_limit = target_undirected_edges * 20 + 1024;
    while (used.size() < target_undirected_edges && guard++ < guard_limit) {
        const uint32_t u = node_dist(rng);
        uint32_t v = node_dist(rng);
        if (u == v) v = (v + 1) % n;
        add_undirected_edge(u, v, adj, used);
    }

    CsrGraph g;
    g.num_nodes = n;
    g.undirected_edges = static_cast<uint32_t>(used.size());
    g.row_ptr.assign(n + 1, 0);

    uint64_t directed_edges = 0;
    for (uint32_t u = 0; u < n; ++u) {
        auto& row = adj[u];
        std::sort(row.begin(), row.end());
        row.erase(std::unique(row.begin(), row.end()), row.end());
        directed_edges += row.size();
        if (directed_edges > static_cast<uint64_t>(std::numeric_limits<int32_t>::max())) {
            throw std::runtime_error("directed edge count exceeds int32_t range");
        }
        g.row_ptr[u + 1] = static_cast<int32_t>(directed_edges);
    }

    g.col_idx.reserve(static_cast<size_t>(directed_edges));
    for (uint32_t u = 0; u < n; ++u) {
        for (int32_t v : adj[u]) g.col_idx.push_back(v);
    }

    g.status.assign(n, 0);
    std::uniform_int_distribution<uint32_t> pct_dist(1, 100);
    for (uint32_t u = 0; u < n; ++u) {
        const bool healthy = (pct_dist(rng) <= cfg.health_percent);
        g.status[u] = healthy ? 1 : 0;
    }

    // The fault node is always unavailable. For performance-oriented experiments,
    // fault_radius can additionally make every node within the given BFS radius
    // unavailable. This prevents the traversal from stopping at L0/L1 simply
    // because a random one-hop neighbor happens to be healthy.
    if (cfg.fault_radius > 0) {
        const auto near_fault = collect_nodes_within_radius(adj, cfg.fault_node, cfg.fault_radius);
        for (uint32_t u = 0; u < n; ++u) {
            if (near_fault[u] >= 0 && static_cast<uint32_t>(near_fault[u]) <= cfg.fault_radius) {
                g.status[u] = 0;
            }
        }
    } else {
        g.status[cfg.fault_node] = 0;
    }

    // Keep at least one replaceable node outside the forced unavailable radius
    // whenever the graph contains such a node. The chosen fallback is deterministic,
    // so repeated tests with the same topology are stable.
    bool has_healthy = false;
    for (int32_t s : g.status) {
        if (s == 1) {
            has_healthy = true;
            break;
        }
    }
    if (!has_healthy && n > 1) {
        auto near_fault = collect_nodes_within_radius(adj, cfg.fault_node, cfg.fault_radius);
        for (uint32_t step = 1; step < n; ++step) {
            const uint32_t candidate = (cfg.fault_node + step) % n;
            if (candidate != cfg.fault_node &&
                (cfg.fault_radius == 0 || near_fault[candidate] < 0 ||
                 static_cast<uint32_t>(near_fault[candidate]) > cfg.fault_radius)) {
                g.status[candidate] = 1;
                break;
            }
        }
    }

    (void)timer;
    return g;
}


#### 5.1.3 串行广度优先遍历

串行参考逐层扩展当前结点集合，记录访问层次、处理边数和各层规模。当某一层首次发现健康结点时，从该层选择编号最小的候选并结束搜索。


In [ ]:
%%writefile -a Source/02.01/src/network_bfs_cpu.cpp

BfsResult serial_bfs_reference(const CsrGraph& g, uint32_t fault_node, uint32_t max_depth) {
    if (fault_node >= g.num_nodes) {
        throw std::invalid_argument("fault_node must be in [0, num_nodes)");
    }
    if (g.row_ptr.size() != static_cast<size_t>(g.num_nodes) + 1 ||
        g.status.size() != g.num_nodes) {
        throw std::invalid_argument("invalid CSR graph");
    }
    const uint32_t depth_limit = max_depth == 0 ? g.num_nodes : max_depth;

    BfsResult result;
    result.level.assign(g.num_nodes, kUnvisited);
    std::vector<int32_t> visited(g.num_nodes, 0);
    std::vector<int32_t> frontier;
    frontier.reserve(g.num_nodes);
    frontier.push_back(static_cast<int32_t>(fault_node));
    visited[fault_node] = 1;
    result.level[fault_node] = 0;
    result.visited_count = 1;

    Timer timer;
    for (uint32_t depth = 0; !frontier.empty() && depth < depth_limit; ++depth) {
        result.frontier_sizes.push_back(static_cast<uint32_t>(frontier.size()));
        uint64_t edges_this_level = 0;
        std::vector<int32_t> next;
        next.reserve(frontier.size() * 2);
        int32_t best_candidate = kDeviceInvalidCandidate;

        for (int32_t u : frontier) {
            if (u < 0 || static_cast<uint32_t>(u) >= g.num_nodes) continue;
            const int32_t begin = g.row_ptr[static_cast<size_t>(u)];
            const int32_t end = g.row_ptr[static_cast<size_t>(u) + 1];
            edges_this_level += static_cast<uint64_t>(std::max(0, end - begin));
            for (int32_t p = begin; p < end; ++p) {
                const int32_t v = g.col_idx[static_cast<size_t>(p)];
                if (v < 0 || static_cast<uint32_t>(v) >= g.num_nodes) continue;
                if (visited[static_cast<size_t>(v)] == 0) {
                    visited[static_cast<size_t>(v)] = 1;
                    result.level[static_cast<size_t>(v)] = static_cast<int32_t>(depth + 1);
                    ++result.visited_count;
                    next.push_back(v);
                    if (g.status[static_cast<size_t>(v)] == 1 && v < best_candidate) {
                        best_candidate = v;
                    }
                }
            }
        }

        result.processed_edges += edges_this_level;
        result.frontier_edges.push_back(edges_this_level);
        if (best_candidate != kDeviceInvalidCandidate) {
            result.replacement_node = best_candidate;
            result.distance = static_cast<int32_t>(depth + 1);
            break;
        }
        frontier.swap(next);
    }
    result.timing.bfs_total_us = timer.elapsed_us();
    return result;
}

void dump_csr_small(const CsrGraph& g, uint32_t max_nodes) {
    const uint32_t n = std::min(g.num_nodes, max_nodes);
    std::cout << "CSR sample(first " << n << " nodes):\n";
    for (uint32_t u = 0; u < n; ++u) {
        std::cout << "  node=" << u << " status=" << g.status[u] << " neighbors=[";
        const int32_t begin = g.row_ptr[u];
        const int32_t end = g.row_ptr[u + 1];
        for (int32_t p = begin; p < end; ++p) {
            if (p > begin) std::cout << ",";
            std::cout << g.col_idx[static_cast<size_t>(p)];
        }
        std::cout << "]\n";
    }
}

}  // namespace netbfs


### 5.2 Host侧算子调用

#### 5.2.1 参数解析与数据填充

Host程序接收网络规模、拓扑特征、故障条件、分块大小、并行核心数和测量次数等参数。为了满足整块搬运要求，邻接数据和当前层数据在末尾预留安全填充区域。


In [ ]:
%%writefile Source/02.01/ascend_ops/host_launch/bfs_npu_main.cpp

#include <acl/acl.h>
#include <aclrtlaunch_bfs_expand_level.h>

#include "network_bfs_cpu.h"

#include <cstdint>
#include <cstdlib>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                                  \
    do {                                                                                 \
        aclError _ret = (expr);                                                         \
        if (_ret != ACL_SUCCESS) {                                                      \
            throw std::runtime_error(std::string("ACL error: ") + #expr +              \
                                     ", code=" + std::to_string(static_cast<int>(_ret))); \
        }                                                                                \
    } while (0)

namespace {

struct Config : public netbfs::BfsConfig {
    int32_t device = 0;
    std::string csv_path = "results/network_bfs_result.csv";
};

void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "Options:\n"
              << "  --device <id>                device id, default: 0\n"
              << "  --nodes <num>                node count, default: 262144\n"
              << "  --avg-degree <num>           target average undirected degree, default: 32\n"
              << "  --fault-node <id>            source faulty node, default: 0\n"
              << "  --health-percent <0..100>    probability of healthy nodes, default: 1\n"
              << "  --fault-radius <num>         force nodes within this BFS radius unavailable, default: 4\n"
              << "  --frontier-block-len <num>   static Tensor frontier block length, default: 32\n"
              << "  --neighbor-tile-len <num>    static Tensor neighbor tile length, default: 32\n"
              << "  --block-dim <num>            AI Core launch blockDim, default: 4\n"
              << "  --warmup <num>               warmup count, default: 2\n"
              << "  --repeat <num>               repeat count, default: 10\n"
              << "  --seed <num>                 random seed, default: 1234\n"
              << "  --csv <path>                 result CSV path, default: results/network_bfs_result.csv\n"
              << "  --max-depth <num>            BFS depth limit, default: nodes\n"
              << "  --sweep                      test frontier block len 256/512/1024/2048\n"
              << "  --print-graph                print a small CSR sample\n"
              << "  --print-levels               print first 32 output levels\n"
              << "  -h, --help                   show help\n";
}

Config parse_args(int argc, char** argv) {
    Config cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + name);
            return argv[++i];
        };
        if (arg == "--device") {
            cfg.device = std::stoi(need_value(arg));
        } else if (arg == "--nodes") {
            cfg.num_nodes = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--avg-degree") {
            cfg.avg_degree = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-node") {
            cfg.fault_node = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--health-percent") {
            cfg.health_percent = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-radius") {
            cfg.fault_radius = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--frontier-block-len") {
            cfg.frontier_block_len = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--neighbor-tile-len") {
            cfg.neighbor_tile_len = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--block-dim") {
            cfg.block_dim = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--seed") {
            cfg.seed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--csv") {
            cfg.csv_path = need_value(arg);
        } else if (arg == "--max-depth") {
            cfg.max_depth = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--sweep") {
            cfg.sweep = true;
        } else if (arg == "--print-graph") {
            cfg.print_graph = true;
        } else if (arg == "--print-levels") {
            cfg.print_levels = true;
        } else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return cfg;
}

std::vector<int32_t> make_padded_colidx(const netbfs::CsrGraph& g, uint32_t neighborTileLen) {
    const uint32_t padded = netbfs::round_up(static_cast<uint32_t>(g.col_idx.size()) + neighborTileLen, neighborTileLen);
    std::vector<int32_t> out(padded, -1);
    std::copy(g.col_idx.begin(), g.col_idx.end(), out.begin());
    return out;
}

std::vector<int32_t> make_padded_frontier(const std::vector<int32_t>& frontier, uint32_t capacity, uint32_t blockLen) {
    const uint32_t active = static_cast<uint32_t>(frontier.size());
    const uint32_t padded = std::max<uint32_t>(blockLen, netbfs::round_up(std::max(active, 1u), blockLen));
    if (padded > capacity) {
        throw std::runtime_error("frontier capacity exceeded");
    }
    std::vector<int32_t> out(padded, -1);
    std::copy(frontier.begin(), frontier.end(), out.begin());
    return out;
}

uint32_t count_visited(const std::vector<int32_t>& level) {
    uint32_t n = 0;
    for (int32_t v : level) n += (v >= 0 ? 1u : 0u);
    return n;
}

bool is_valid_device_candidate(const netbfs::CsrGraph& graph, uint32_t faultNode, int32_t v) {
    if (v < 0) return false;
    const uint32_t vu = static_cast<uint32_t>(v);
    if (vu >= graph.num_nodes) return false;
    if (vu == faultNode) return false;
    return graph.status[vu] == 1;
}


#### 5.2.2 Device内存管理

Host为网络拓扑、结点状态、当前层数据、发现位图和统计结果分别申请Device内存。实验结束后统一释放这些资源，避免多次运行时累积占用。


In [ ]:
%%writefile -a Source/02.01/ascend_ops/host_launch/bfs_npu_main.cpp

struct DeviceBuffers {
    void* rowPtr = nullptr;
    void* colIdx = nullptr;
    void* status = nullptr;
    void* frontier = nullptr;
    void* visited = nullptr;
    void* level = nullptr;
    void* nextBitmap = nullptr;
    void* candidate = nullptr;
    void* edgeCount = nullptr;
};

void free_buffers(DeviceBuffers& b) {
    if (b.edgeCount) aclrtFree(b.edgeCount);
    if (b.candidate) aclrtFree(b.candidate);
    if (b.nextBitmap) aclrtFree(b.nextBitmap);
    if (b.level) aclrtFree(b.level);
    if (b.visited) aclrtFree(b.visited);
    if (b.frontier) aclrtFree(b.frontier);
    if (b.status) aclrtFree(b.status);
    if (b.colIdx) aclrtFree(b.colIdx);
    if (b.rowPtr) aclrtFree(b.rowPtr);
    b = DeviceBuffers{};
}


#### 5.2.3 单次遍历初始化

每次遍历开始时，Host清空访问状态和层次数组，将故障结点作为第0层的唯一结点，并准备本次运行所需的位图与临时结果空间。


In [ ]:
%%writefile -a Source/02.01/ascend_ops/host_launch/bfs_npu_main.cpp

netbfs::BfsResult run_device_bfs_once(const Config& cfg,
                                    const netbfs::CsrGraph& graph,
                                    const std::vector<int32_t>& colIdxPadded,
                                    const netbfs::BfsResult& ref,
                                    DeviceBuffers& dev,
                                    aclrtStream stream,
                                    bool timed) {
    const uint32_t n = graph.num_nodes;
    const uint32_t numEdgesPadded = static_cast<uint32_t>(colIdxPadded.size());
    const uint32_t frontierCapacity = netbfs::round_up(n + cfg.frontier_block_len, cfg.frontier_block_len);
    const size_t nodeBytes = static_cast<size_t>(n) * sizeof(int32_t);
    const size_t frontierBytes = static_cast<size_t>(frontierCapacity) * sizeof(int32_t);
    const uint32_t wordsPerBitmap = (n + 31u) >> 5;
    const size_t nextBitmapBytes = static_cast<size_t>(cfg.block_dim) * wordsPerBitmap * sizeof(uint32_t);

    std::vector<int32_t> visitedHost(n, 0);
    std::vector<int32_t> levelHost(n, netbfs::kUnvisited);
    visitedHost[cfg.fault_node] = 1;
    levelHost[cfg.fault_node] = 0;


#### 5.2.4 逐层调度与Host归并

每一层先把当前结点集合传到Device并清空发现位图，然后启动多核层次扩展算子。算子完成后，Host回传各核心位图并进行归并。

Host根据归并结果更新访问状态、层次和下一层结点集合，同时完成健康候选判定和本层处理边数统计。


In [ ]:
%%writefile -a Source/02.01/ascend_ops/host_launch/bfs_npu_main.cpp

    // Host keeps visited/level as authoritative state. The Device expands
    // the current frontier and writes per-core compact bitsets. This keeps the
    // race-free layer semantics of per-core discovery while reducing transfer
    // volume by 32x compared with int32 per-core bitmaps.

    netbfs::BfsResult result;
    std::vector<int32_t> frontier{static_cast<int32_t>(cfg.fault_node)};
    std::vector<uint32_t> nextBitmap(static_cast<size_t>(cfg.block_dim) * wordsPerBitmap, 0);
    std::vector<uint32_t> mergedBitmap(wordsPerBitmap, 0);

    netbfs::Timer totalTimer;
    double kernelUs = 0.0;
    double frontierUs = 0.0;
    const uint32_t depthLimit = netbfs::effective_max_depth(cfg);

    for (uint32_t depth = 0; !frontier.empty() && depth < depthLimit; ++depth) {
        result.frontier_sizes.push_back(static_cast<uint32_t>(frontier.size()));

        netbfs::Timer frontierTimer;
        auto paddedFrontier = make_padded_frontier(frontier, frontierCapacity, cfg.frontier_block_len);
        const uint32_t frontierPaddedCount = static_cast<uint32_t>(paddedFrontier.size());
        const size_t paddedFrontierBytes = static_cast<size_t>(frontierPaddedCount) * sizeof(int32_t);
        ACL_CHECK(aclrtMemcpy(dev.frontier, frontierBytes, paddedFrontier.data(), paddedFrontierBytes, ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemset(dev.nextBitmap, nextBitmapBytes, 0, nextBitmapBytes));
        if (timed) frontierUs += frontierTimer.elapsed_us();

        netbfs::Timer kernelTimer;
        ACLRT_LAUNCH_KERNEL(bfs_expand_level)(cfg.block_dim, stream,
                                              dev.rowPtr, dev.colIdx, dev.status,
                                              dev.frontier, dev.visited, dev.level,
                                              dev.nextBitmap, dev.candidate, dev.edgeCount,
                                              n, numEdgesPadded,
                                              static_cast<uint32_t>(frontier.size()), frontierPaddedCount,
                                              depth, cfg.frontier_block_len, cfg.neighbor_tile_len,
                                              cfg.block_dim, cfg.fault_node);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        if (timed) kernelUs += kernelTimer.elapsed_us();

        netbfs::Timer postTimer;

        uint64_t edgesThisLevel = 0;
        for (int32_t u : frontier) {
            if (u < 0 || static_cast<uint32_t>(u) >= n) continue;
            const int32_t begin = graph.row_ptr[static_cast<size_t>(u)];
            const int32_t end = graph.row_ptr[static_cast<size_t>(u) + 1];
            if (end > begin) edgesThisLevel += static_cast<uint64_t>(end - begin);
        }
        result.frontier_edges.push_back(edgesThisLevel);
        result.processed_edges += edgesThisLevel;

        ACL_CHECK(aclrtMemcpy(nextBitmap.data(), nextBitmapBytes, dev.nextBitmap,
                              nextBitmapBytes, ACL_MEMCPY_DEVICE_TO_HOST));

        std::fill(mergedBitmap.begin(), mergedBitmap.end(), 0u);
        for (uint32_t c = 0; c < cfg.block_dim; ++c) {
            const size_t base = static_cast<size_t>(c) * wordsPerBitmap;
            for (uint32_t w = 0; w < wordsPerBitmap; ++w) {
                mergedBitmap[w] |= nextBitmap[base + w];
            }
        }

        int32_t best = netbfs::kDeviceInvalidCandidate;
        std::vector<int32_t> next;
        next.reserve(std::min<size_t>(n, std::max<size_t>(frontier.size() * 2, 64)));
        for (uint32_t w = 0; w < wordsPerBitmap; ++w) {
            uint32_t word = mergedBitmap[w];
            while (word != 0u) {
                const uint32_t bit = static_cast<uint32_t>(__builtin_ctz(word));
                const uint32_t v = (w << 5) + bit;
                word &= (word - 1u);
                if (v >= n || visitedHost[v] != 0) {
                    continue;
                }
                visitedHost[v] = 1;
                levelHost[v] = static_cast<int32_t>(depth + 1);
                next.push_back(static_cast<int32_t>(v));
                if (is_valid_device_candidate(graph, cfg.fault_node, static_cast<int32_t>(v)) &&
                    static_cast<int32_t>(v) < best) {
                    best = static_cast<int32_t>(v);
                }
            }
        }

        if (best != netbfs::kDeviceInvalidCandidate) {
            result.replacement_node = best;
            result.distance = static_cast<int32_t>(depth + 1);
            if (timed) frontierUs += postTimer.elapsed_us();
            break;
        }

        frontier.swap(next);
        if (timed) frontierUs += postTimer.elapsed_us();
    }

    result.timing.bfs_total_us = timed ? totalTimer.elapsed_us() : 0.0;
    result.timing.kernel_us = kernelUs;
    result.timing.host_frontier_us = frontierUs;
    result.level = levelHost;
    result.visited_count = count_visited(result.level);

    (void)ref;
    return result;
}


#### 5.2.5 重复测量与结果记录

Host上传固定不变的网络数据后，先执行预热，再进行多次正式测量并计算平均耗时。最后将Device结果与CPU参考比较，并把运行参数、正确性和性能指标写入CSV。


In [ ]:
%%writefile -a Source/02.01/ascend_ops/host_launch/bfs_npu_main.cpp

void reset_csv(const std::string& path) {
    std::ofstream out(path);
    if (!out) {
        throw std::runtime_error("failed to open csv: " + path);
    }
    out << "implementation,device,nodes,directed_edges,undirected_edges,avg_degree,fault_node,"
           "health_percent,fault_radius,frontier_block_len,neighbor_tile_len,block_dim,warmup,repeat,"
           "seed,max_depth,replacement_node,distance,visited_count,processed_edges,preprocess_us,"
           "serial_bfs_us,kernel_us,frontier_us,total_us,medges_per_s,speedup,status,replacement_ok,"
           "distance_ok,level_ok,level_mismatch\n";
    if (!out) {
        throw std::runtime_error("failed to write csv header: " + path);
    }
}

void append_csv_row(const std::string& path,
                    const Config& cfg,
                    const netbfs::CsrGraph& graph,
                    const netbfs::BfsResult& ref,
                    const netbfs::BfsResult& result,
                    const netbfs::CheckResult& check,
                    const std::string& status) {
    std::ofstream out(path, std::ios::app);
    if (!out) {
        throw std::runtime_error("failed to append csv: " + path);
    }
    const double medgesPerSecond =
        netbfs::edges_per_second(result.processed_edges, result.timing.bfs_total_us) / 1e6;
    const double speedup = result.timing.bfs_total_us > 0.0
        ? ref.timing.bfs_total_us / result.timing.bfs_total_us
        : 0.0;
    out << "host_merged_per_core_bitset" << ','
        << cfg.device << ',' << graph.num_nodes << ',' << graph.col_idx.size() << ','
        << graph.undirected_edges << ',' << cfg.avg_degree << ',' << cfg.fault_node << ','
        << cfg.health_percent << ',' << cfg.fault_radius << ',' << cfg.frontier_block_len << ','
        << cfg.neighbor_tile_len << ',' << cfg.block_dim << ',' << cfg.warmup << ','
        << cfg.repeat << ',' << cfg.seed << ',' << netbfs::effective_max_depth(cfg) << ','
        << result.replacement_node << ',' << result.distance << ',' << result.visited_count << ','
        << result.processed_edges << ',' << std::fixed << std::setprecision(3)
        << ref.timing.preprocess_us << ',' << ref.timing.bfs_total_us << ','
        << result.timing.kernel_us << ',' << result.timing.host_frontier_us << ','
        << result.timing.bfs_total_us << ',' << std::setprecision(6)
        << medgesPerSecond << ',' << speedup << ',' << status << ','
        << check.replacement_ok << ',' << check.distance_ok << ',' << check.level_ok << ','
        << check.level_mismatch << '\n';
    if (!out) {
        throw std::runtime_error("failed to write csv row: " + path);
    }
}

void run_one(const Config& cfg, const netbfs::CsrGraph& graph, const netbfs::BfsResult& ref) {
    netbfs::check_config(cfg);

    const uint32_t n = graph.num_nodes;
    const auto colIdxPadded = make_padded_colidx(graph, cfg.neighbor_tile_len);
    const uint32_t frontierCapacity = netbfs::round_up(n + cfg.frontier_block_len, cfg.frontier_block_len);

    DeviceBuffers dev;
    aclrtStream stream = nullptr;

    const size_t rowPtrBytes = static_cast<size_t>(n + 1) * sizeof(int32_t);
    const size_t colIdxBytes = static_cast<size_t>(colIdxPadded.size()) * sizeof(int32_t);
    const size_t nodeBytes = static_cast<size_t>(n) * sizeof(int32_t);
    const size_t frontierBytes = static_cast<size_t>(frontierCapacity) * sizeof(int32_t);
    const uint32_t wordsPerBitmap = (n + 31u) >> 5;
    const size_t nextBitmapBytes = static_cast<size_t>(cfg.block_dim) * wordsPerBitmap * sizeof(uint32_t);
    const size_t candidateBytes = static_cast<size_t>(cfg.block_dim) * sizeof(int32_t);
    const size_t edgeCountBytes = static_cast<size_t>(cfg.block_dim) * sizeof(int32_t);

    ACL_CHECK(aclrtCreateStream(&stream));
    ACL_CHECK(aclrtMalloc(&dev.rowPtr, rowPtrBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.colIdx, colIdxBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.status, nodeBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.frontier, frontierBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.visited, nodeBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.level, nodeBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.nextBitmap, nextBitmapBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.candidate, candidateBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.edgeCount, edgeCountBytes, ACL_MEM_MALLOC_HUGE_FIRST));

    ACL_CHECK(aclrtMemcpy(dev.rowPtr, rowPtrBytes, graph.row_ptr.data(), rowPtrBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(dev.colIdx, colIdxBytes, colIdxPadded.data(), colIdxBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(dev.status, nodeBytes, graph.status.data(), nodeBytes, ACL_MEMCPY_HOST_TO_DEVICE));

    for (uint32_t i = 0; i < cfg.warmup; ++i) {
        (void)run_device_bfs_once(cfg, graph, colIdxPadded, ref, dev, stream, false);
    }

    netbfs::BfsResult last;
    netbfs::TimingUs avg;
    const uint32_t repeat = std::max(1u, cfg.repeat);
    for (uint32_t i = 0; i < repeat; ++i) {
        last = run_device_bfs_once(cfg, graph, colIdxPadded, ref, dev, stream, true);
        avg.bfs_total_us += last.timing.bfs_total_us;
        avg.kernel_us += last.timing.kernel_us;
        avg.host_frontier_us += last.timing.host_frontier_us;
    }
    avg.bfs_total_us /= repeat;
    avg.kernel_us /= repeat;
    avg.host_frontier_us /= repeat;
    last.timing = avg;

    const auto check = netbfs::compare_results(last, ref);
    const std::string status = (check.replacement_ok && check.distance_ok && check.level_ok) ? "PASS" : "FAIL";
    netbfs::print_result_row(graph, cfg.fault_node, last, status, true);
    std::cout << "  implementation=host_merged_per_core_bitset"
              << ", launchBlockDim=" << cfg.block_dim
              << ", frontierBlockLen=" << cfg.frontier_block_len
              << ", neighborTileLen=" << cfg.neighbor_tile_len
              << ", warmup=" << cfg.warmup
              << ", repeat=" << cfg.repeat
              << ", faultRadius=" << cfg.fault_radius << "\n";
    std::cout << "  check: replacement_ok=" << check.replacement_ok
              << ", distance_ok=" << check.distance_ok
              << ", level_ok=" << check.level_ok
              << ", level_mismatch=" << check.level_mismatch << "\n";
    std::cout << "  frontier sizes:";
    for (size_t i = 0; i < last.frontier_sizes.size(); ++i) {
        std::cout << " L" << i << "=" << last.frontier_sizes[i]
                  << "(E=" << last.frontier_edges[i] << ")";
    }
    std::cout << "\n";
    if (cfg.print_levels) netbfs::print_levels_sample(last);
    free_buffers(dev);
    aclrtDestroyStream(stream);
    append_csv_row(cfg.csv_path, cfg, graph, ref, last, check, status);
}


#### 5.2.6 主流程与运行时管理

主流程依次完成网络数据生成、CPU参考计算、运行时初始化、Device实验、结果输出和资源释放。正常结束或发生错误时，程序都会按既定顺序关闭Device和运行时环境。


In [ ]:
%%writefile -a Source/02.01/ascend_ops/host_launch/bfs_npu_main.cpp

}  // namespace

int main(int argc, char** argv) {
    try {
        const Config cfg = parse_args(argc, argv);
        netbfs::check_config(cfg);

        netbfs::Timer prepTimer;
        const auto graph = netbfs::make_network_graph(cfg);
        const double prepUs = prepTimer.elapsed_us();
        auto ref = netbfs::serial_bfs_reference(graph, cfg.fault_node, netbfs::effective_max_depth(cfg));
        ref.timing.preprocess_us = prepUs;
        reset_csv(cfg.csv_path);

        if (cfg.print_graph) netbfs::dump_csr_small(graph);
        netbfs::print_graph_summary(graph, cfg.fault_node);
        std::cout << "reference: replacement=" << ref.replacement_node
                  << ", distance=" << ref.distance
                  << ", visited=" << ref.visited_count
                  << ", processed_edges=" << ref.processed_edges
                  << ", preprocess_us=" << std::fixed << std::setprecision(2) << prepUs
                  << ", serial_bfs_us=" << ref.timing.bfs_total_us << std::defaultfloat << "\n";

        ACL_CHECK(aclInit(nullptr));
        ACL_CHECK(aclrtSetDevice(cfg.device));

        netbfs::print_result_header(true);
        if (cfg.sweep) {
            for (uint32_t len : {256u, 512u, 1024u, 2048u}) {
                Config cur = cfg;
                cur.frontier_block_len = len;
                run_one(cur, graph, ref);
            }
        } else {
            run_one(cfg, graph, ref);
        }

        std::cout << "[info] csv written to " << cfg.csv_path << "\n";

        ACL_CHECK(aclrtResetDevice(cfg.device));
        ACL_CHECK(aclFinalize());
        return 0;
    } catch (const std::exception& e) {
        std::cerr << "error: " << e.what() << "\n";
        usage(argv[0]);
        return 1;
    }
}


### 5.3 工程构建

工程构建过程先编译CPU参考库，再编译Ascend C核函数和NPU Host程序，最后链接运行时依赖。构建配置会根据当前Ascend安装位置选择对应工具链。


In [ ]:
%%writefile Source/02.01/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
project(ascendc_parallel_bfs_diagnosis LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build Ascend C NPU demo" OFF)

add_library(network_bfs_cpu
    src/network_bfs_cpu.cpp
)
target_include_directories(network_bfs_cpu PUBLIC include)
target_compile_options(network_bfs_cpu PRIVATE -Wall -Wextra -Wpedantic)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode: npu/cpu/sim")
  set(SOC_VERSION "ascend910b1" CACHE STRING "Ascend SOC version, e.g. ascend910b1/ascend910b2/ascend310p3")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH)
    set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation path" FORCE)
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. Check ASCEND_CANN_PATH/ASCEND_INSTALL_PATH.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(network_bfs_kernels STATIC
      ascend_ops/op_kernel/bfs_expand_level.cpp
  )
  ascendc_include_directories(network_bfs_kernels PRIVATE
      ${CMAKE_CURRENT_SOURCE_DIR}/ascend_ops/op_host
  )
  ascendc_compile_definitions(network_bfs_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  add_executable(network_bfs_ascend_demo
      ascend_ops/host_launch/bfs_npu_main.cpp
  )
  target_include_directories(network_bfs_ascend_demo PRIVATE
      include
      ascend_ops/op_host
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${CMAKE_INSTALL_PREFIX}/include/network_bfs_kernels
      ${CMAKE_BINARY_DIR}/out/include/network_bfs_kernels
  )
  target_link_directories(network_bfs_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
  )
  target_link_libraries(network_bfs_ascend_demo PRIVATE
      network_bfs_kernels
      network_bfs_cpu
      ascendcl
  )
  add_dependencies(network_bfs_ascend_demo network_bfs_kernels)
endif()

install(DIRECTORY ascend_ops DESTINATION share/ascendc_parallel_bfs_diagnosis)


### 5.4 构建、运行与性能采集脚本

运行脚本负责加载环境、配置工程、编译程序、传递实验参数并保存CSV结果。性能采集模板可在需要时记录更细粒度的核函数和调度信息。


In [ ]:
%%writefile Source/02.01/scripts/run_ascend.sh

#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
RESULT_DIR="${SCRIPT_DIR}/results"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"
if [[ -z "${ASCEND_INSTALL_PATH:-}" ]]; then
  for candidate in /opt/conda/Ascend/cann-* "${ASCEND_INSTALL_PATH_DEFAULT}"; do
    if [[ -d "${candidate}" ]]; then
      ASCEND_INSTALL_PATH="${candidate}"
      break
    fi
  done
fi
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-${ASCEND_INSTALL_PATH_DEFAULT}}"
SOC_VERSION="${SOC_VERSION:-ascend910b1}"
DEVICE_ID=0
RUN_MODE="npu"
BUILD_TYPE="Release"
CLEAN=0
NODES=262144
AVG_DEGREE=32
FAULT_NODE=0
HEALTH_PERCENT=1
FAULT_RADIUS=4
FRONTIER_BLOCK_LEN=32
NEIGHBOR_TILE_LEN=32
BLOCK_DIM=4
WARMUP=2
REPEAT=10
SEED=1234
MAX_DEPTH=0
SWEEP=0
EXTRA_ARGS=()

usage() {
  cat <<USAGE
Usage: bash scripts/run_ascend.sh [options] [-- extra_args_for_binary]

Options:
  -a <path>   ASCEND_INSTALL_PATH, default: /usr/local/Ascend/ascend-toolkit/latest
  -v <soc>    SOC_VERSION, default: ascend910b1. Example: ascend910b2, ascend910b3, ascend310p3
  -d <id>     device id, default: 0
  -n <num>    graph node count, default: 262144
  -g <num>    target average degree, default: 32
  -f <id>     fault source node id, default: 0
  -p <num>    healthy node percentage, default: 1
  -q <num>    forced unavailable BFS radius around fault node, default: 4
  -l <num>    frontier static Tensor block length, default: 32
  -e <num>    neighbor static Tensor tile length, default: 32
  -b <num>    AI Core launch blockDim, default: 4
  -w <num>    warmup count, default: 2
  -r <num>    repeat count, default: 10
  -m <mode>   CMake run mode, default: npu
  -t <type>   CMake build type, default: Release
  -x <num>    max BFS depth, default: nodes
  -s          sweep frontier block length = 256/512/1024/2048
  -c          clean build directory before building
  -h          show help

Examples:
  bash scripts/run_ascend.sh
  bash scripts/run_ascend.sh -n 16384 -g 12 -f 7 -q 2 -l 512 -e 512 -b 8 -w 2 -r 10
  bash scripts/run_ascend.sh -s -n 8192
  bash scripts/run_ascend.sh -- --print-levels
USAGE
}

while getopts ":a:v:d:n:g:f:p:q:l:e:b:w:r:m:t:x:sch" opt; do
  case ${opt} in
    a) ASCEND_INSTALL_PATH="${OPTARG}" ;;
    v) SOC_VERSION="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    n) NODES="${OPTARG}" ;;
    g) AVG_DEGREE="${OPTARG}" ;;
    f) FAULT_NODE="${OPTARG}" ;;
    p) HEALTH_PERCENT="${OPTARG}" ;;
    q) FAULT_RADIUS="${OPTARG}" ;;
    l) FRONTIER_BLOCK_LEN="${OPTARG}" ;;
    e) NEIGHBOR_TILE_LEN="${OPTARG}" ;;
    b) BLOCK_DIM="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    m) RUN_MODE="${OPTARG}" ;;
    t) BUILD_TYPE="${OPTARG}" ;;
    x) MAX_DEPTH="${OPTARG}" ;;
    s) SWEEP=1 ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))

if [[ $# -gt 0 && "$1" == "--" ]]; then
  shift
fi
EXTRA_ARGS=("$@")

if [[ ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  echo "ASCEND_INSTALL_PATH does not exist: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

if [[ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  # shellcheck disable=SC1090
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
fi

export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
export SOC_VERSION

if [[ "${CLEAN}" == "1" ]]; then
  rm -rf "${BUILD_DIR}"
fi
mkdir -p "${BUILD_DIR}" "${RESULT_DIR}"
cd "${BUILD_DIR}"

cmake "${SCRIPT_DIR}" \
  -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${ASCEND_INSTALL_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE="${RUN_MODE}"

cmake --build . -j

BIN="${BUILD_DIR}/network_bfs_ascend_demo"
if [[ ! -x "${BIN}" ]]; then
  echo "Cannot find executable: ${BIN}" >&2
  exit 1
fi

CSV="${RESULT_DIR}/network_bfs_result.csv"
CMD=("${BIN}" --device "${DEVICE_ID}" --nodes "${NODES}" --avg-degree "${AVG_DEGREE}" \
     --fault-node "${FAULT_NODE}" --health-percent "${HEALTH_PERCENT}" --fault-radius "${FAULT_RADIUS}" \
     --frontier-block-len "${FRONTIER_BLOCK_LEN}" --neighbor-tile-len "${NEIGHBOR_TILE_LEN}" \
     --block-dim "${BLOCK_DIM}" --warmup "${WARMUP}" --repeat "${REPEAT}" --seed "${SEED}" \
     --csv "${CSV}")
if [[ "${MAX_DEPTH}" != "0" ]]; then
  CMD+=(--max-depth "${MAX_DEPTH}")
fi
if [[ "${SWEEP}" == "1" ]]; then
  CMD+=(--sweep)
fi
CMD+=("${EXTRA_ARGS[@]}")

echo "[RUN] ${CMD[*]}"
"${CMD[@]}"


In [ ]:
%%writefile Source/02.01/scripts/profile_msprof_template.sh

#!/usr/bin/env bash
set -euo pipefail
# Template. Replace SOC/device/options as needed on the Ascend host.
msprof --application="./build_ascend/network_bfs_ascend_demo --device 0 --nodes 262144 --avg-degree 32 --fault-node 0 --health-percent 1 --fault-radius 4 --frontier-block-len 32 --neighbor-tile-len 32 --block-dim 4 --warmup 2 --repeat 10 --seed 1234" \
       --output=./msprof_network_bfs


In [ ]:
!chmod +x Source/02.01/scripts/run_ascend.sh
!chmod +x Source/02.01/scripts/profile_msprof_template.sh
!find Source/02.01 -maxdepth 4 -type f | sort


### 5.5 实验参数与完整运行

本次实验使用以下参数：

| 参数 | 命令参数 |   默认取值 | 作用                 |
|---|:---:|-------:|--------------------|
| 结点数 | `-n` | 262144 | 决定网络、访问状态和位图规模     |
| 平均度 | `-g` |     32 | 控制每个结点的平均连接数量      |
| 故障结点 | `-f` |      0 | 作为广度优先遍历的起点        |
| 健康比例 | `-p` |     1% | 控制可作为替换对象的结点数量     |
| 不可用范围 | `-q` |     4跳 | 使搜索经过多个层次后再出现候选结点  |
| 前沿块长度 | `-l` |     32 | 控制一次处理的当前层结点数量     |
| 邻接片段长度 | `-e` |     32 | 控制一次处理的邻接结点数量      |
| 并行核心数 | `-b` |      4 | 控制参与邻接扩展的AI Core数量 |
| 预热次数 | `-w` |      2 | 降低首次启动对计时的影响       |
| 重复次数 | `-r` |     10 | 多次测量后计算平均耗时        |
| 随机种子 | `--seed` |   1234 | 保证网络拓扑和结点状态可以重复    |

随机种子由运行脚本固定为1234，默认命令不需要重复输入；需要修改时，可以在命令末尾通过`-- --seed <数值>`传给Host程序。其他参数也设置了默认值，这里在运行命令中显式写出，便于核对每次实验的输入条件。

前沿块和邻接片段都包含32个32位整数，每次有效搬运128字节。分块过小会增加循环和调度次数，分块过大则可能增加无效搬运，因此需要结合图规模和片上存储容量进行选择。

下面按上述参数运行完整实验。程序会先在CPU上计算参考结果，再运行Device算子，最后完成正确性检查、性能统计和CSV保存。


In [ ]:
%%bash
set -euo pipefail
cd Source/02.01
bash scripts/run_ascend.sh -c -n 262144 -g 32 -f 0 -p 1 -q 4 -l 32 -e 32 -b 4 -w 2 -r 10


### 5.6 读取CSV结果

CSV保存每次实验的输入参数、正确性状态、预处理时间、CPU参考时间、Device各阶段耗时、有效边吞吐率和加速比。下面读取并显示本次运行结果。


In [ ]:
import csv
from pathlib import Path

csv_path = Path("Source/02.01/results/network_bfs_result.csv")
if not csv_path.exists():
    print("Result CSV does not exist yet:", csv_path)
    print("Please run the Ascend experiment cell first.")
else:
    with csv_path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        print(row)

    if rows:
        latest = rows[-1]
        print(f"\nStatus: {latest['status']}")
        print(f"CPU serial BFS: {float(latest['serial_bfs_us']):.2f} us")
        print(f"Device total: {float(latest['total_us']):.2f} us")
        print(f"Throughput: {float(latest['medges_per_s']):.3f} MEdges/s")
        print(f"Speedup: {float(latest['speedup']):.2f}x")


### 5.7 结果判读与性能分析

结果分析先看正确性，再看性能。替换结点、最短距离和完整层次数组全部一致时，说明CPU参考与Device计算得到相同的遍历结果。

性能输出分为核函数耗时、层次维护耗时和完整遍历耗时。核函数耗时反映Device并行扩展的开销；层次维护耗时包含位图回传、Host归并和下一层结点集合生成；完整遍历耗时用于与CPU串行BFS比较。

若实际处理边数为$E$，完整遍历耗时为$T_{\mu s}$微秒，则有效边吞吐率为：

$$
\mathrm{Throughput}_{\mathrm{MEdges/s}}=\frac{E}{T_{\mu s}}
$$

CPU与Device的加速比为：

$$
\mathrm{Speedup}=\frac{T_{\mathrm{CPU}}}{T_{\mathrm{Device}}}
$$

预处理时间表示网络生成和CSR转换耗时，不计入BFS加速比。只有正确性检查通过时，耗时、吞吐率和加速比才具有比较意义。


### 5.8 扩展实验

完成默认实验后，可以在262144个服务器结点的规模下开展两组对比。第一组完全采用默认实验参数作为对照，随后保持其余参数不变，分别改变平均连接数、健康服务器比例和故障服务器位置，观察每层待扩展结点数、访问结点数和处理边数。

本实验生成的网络由相邻结点连接和随机链路共同组成，服务器编号本身不表示物理位置。因此，比较不同故障结点时，应根据第0层处理边数以及后续各层结点规模判断该结点更接近连接密集位置还是连接稀疏位置，再解释扩展速度，不能仅凭编号把结点称为中心或边缘。

第二组继续使用同一个默认基准，固定网络规模、随机种子、连接结构、健康状态和故障条件，只改变分块大小或AI Core数量。其中前沿分块长度和邻接片段长度共同组成一组分块设置。分析时先确认正确性通过，再比较Kernel耗时、总耗时和有效边处理吞吐率。分块过小可能增加循环与数据组织开销，分块过大则会增加Local Memory占用，并可能降低不同规模前沿下的适应性。

下面给出需要在终端中手动执行的命令。

当前BFS把所有服务器链路视为等权连接，得到的是最少跳数意义下的替换服务器。如果链路还包含时延、带宽等权重，需要进一步使用带权最短路径方法，并在候选选择时考虑容量约束。还可以继续比较其他网络生成方式或候选优先级规则，但这里仍以CSR组织、层次推进和静态Tensor分块为主要观察内容。


先进入当前Notebook生成的实验目录：

```bash
cd ~
cd Source/02.01
```

运行脚本会自动搜索当前环境中的CANN；若仍提示安装路径不存在，可以先执行`export ASCEND_INSTALL_PATH=/opt/conda/Ascend/cann-9.0.0`。

#### 网络结构与诊断条件对比

Notebook前面的完整运行结果作为统一基准，下面三条命令分别只改变平均连接数、健康服务器比例和故障服务器位置。

```bash
bash scripts/run_ascend.sh -n 262144 -g 16 -f 0      -p 1  -q 4 -l 32 -e 32 -b 4 -w 2 -r 10 -- --csv ../results/ext_degree_16.csv
bash scripts/run_ascend.sh -n 262144 -g 32 -f 0      -p 10 -q 4 -l 32 -e 32 -b 4 -w 2 -r 10 -- --csv ../results/ext_health_10.csv
bash scripts/run_ascend.sh -n 262144 -g 32 -f 131072 -p 1  -q 4 -l 32 -e 32 -b 4 -w 2 -r 10 -- --csv ../results/ext_fault_131072.csv
```

#### 分块与多核对比

下面两条命令继续与默认基准比较：第一条同时调整前沿分块和邻接分块，第二条只调整AI Core数量。

```bash
bash scripts/run_ascend.sh -n 262144 -g 32 -f 0 -p 1 -q 4 -l 128 -e 128 -b 4 -w 2 -r 10 -- --csv ../results/ext_block_128.csv
bash scripts/run_ascend.sh -n 262144 -g 32 -f 0 -p 1 -q 4 -l 32  -e 32  -b 8 -w 2 -r 10 -- --csv ../results/ext_core_8.csv
```

运行后，终端会显示各层结点规模和每层处理边数；各组参数、正确性和性能结果分别保存在`Source/02.01/results/ext_*.csv`中，可在实验结束后统一整理比较。


---
## 6. 实验总结

本实验使用CSR组织网络拓扑，通过静态Tensor对当前层结点和邻接片段进行分块，并由多个AI Core并行完成邻接扩展。各核心使用独立位图记录发现结果，避免并发写冲突；Host统一完成位图归并、层次维护、下一层生成和候选选择。

CPU串行参考用于验证最近健康结点、最短跳数和完整层次结果。预热与重复测量用于获得稳定的性能数据，CSV则统一保存实验参数、正确性和性能指标。
